In [18]:
os.path.dirname

<function posixpath.dirname(p)>

In [16]:
import pandas as pd
import json
import os
import sys

In [20]:
"""
每日板块信息 Dashboard 更新脚本
用法: python3 update_dashboard.py
功能: 读取同目录下的 每日板块信息.xlsx，重新生成 每日板块信息_Dashboard.html
"""
import pandas as pd
import json
import os
import sys

def main():
    # 读取数据
    xlsx = pd.ExcelFile('每日板块信息.xlsx')
    df_index = pd.read_excel(xlsx, sheet_name='指数表现')
    df_valuation = pd.read_excel(xlsx, sheet_name='指数估值').rename(columns={'指数名称）': '指数名称'})
    df_contrib = pd.read_excel(xlsx, sheet_name='指数贡献')
    df_sector = pd.read_excel(xlsx, sheet_name='每日板块信息')

    df_best = df_sector[df_sector['指数名称'] == '沪市最强表现'].head(10)
    df_worst = df_sector[df_sector['指数名称'] == '沪市最弱表现'].head(10)

    data_json = {
        'index': df_index.to_dict(orient='records'),
        'valuation': df_valuation.to_dict(orient='records'),
        'contrib': df_contrib.to_dict(orient='records'),
        'sector': df_sector.to_dict(orient='records'),
        'best': df_best.to_dict(orient='records'),
        'worst': df_worst.to_dict(orient='records'),
    }
    data_str = json.dumps(data_json, ensure_ascii=False, default=str)

    # HTML 模板
    html_template = os.path.join(script_dir, '_dashboard_template.html')
    output_path = os.path.join(script_dir, '每日板块信息_Dashboard.html')

    # 读取模板或直接生成
    template = None
    if os.path.exists(html_template):
        with open(html_template, 'r', encoding='utf-8') as f:
            template = f.read()

    if template and '{DATA_PLACEHOLDER}' in template:
        html = template.replace('{DATA_PLACEHOLDER}', data_str)
    else:
        # 无模板时，读取已有的 HTML 并替换数据
        if os.path.exists(output_path):
            with open(output_path, 'r', encoding='utf-8') as f:
                existing = f.read()
            # 替换 const DATA = ... 部分
            import re
            html = re.sub(
                r'const DATA = \{.*?\};',
                f'const DATA = {data_str};',
                existing,
                flags=re.DOTALL
            )
        else:
            print("❌ 找不到模板文件，请先运行生成脚本创建初始 Dashboard")
            sys.exit(1)

    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(html)

    print(f"✅ Dashboard 已更新: {output_path}")
    print(f"   数据来源: {excel_path}")
    print(f"   指数数量: {len(df_index)}, 板块数量: {len(df_sector)}")



In [21]:

if __name__ == '__main__':
    main()

/opt/anaconda3/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


NameError: name 'script_dir' is not defined